In [ ]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")
print(dataset['train'][0])


In [ ]:

label_names = dataset['train'].features['labels'].feature.names
example_labels = [label_names[i] for i in dataset['train'][0]['labels']]
print(f"Text: {dataset['train'][0]['text']}")
print(f"Detected Emotions: {example_labels}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=28, 
    problem_type="multi_label_classification"
)

# 3. Move to GPU
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded and moved to: {device}")


In [ ]:
import numpy as np

def preprocess_data(examples):
    encoding = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels_matrix = np.zeros((len(examples["labels"]), 28), dtype=np.float32)

    for i, labs in enumerate(examples["labels"]):
        for lab in labs:
            labels_matrix[i, lab] = 1.0
    encoding["labels"] = labels_matrix
    return encoding
tokenized_datasets = dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_datasets.set_format("torch")

sample = tokenized_datasets["train"][0]
print(sample["labels"].dtype)


In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
class FloatLabelTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        if "labels" in inputs:
            inputs["labels"] = inputs["labels"].float()

        outputs = model(**inputs)
        loss = outputs.loss

        return (loss, outputs) if return_outputs else loss
def compute_metrics(eval_pred):

    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")

    try:
        roc_auc = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        roc_auc = float("nan")

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "roc_auc": roc_auc
    }
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,

    weight_decay=0.01,
    logging_steps=25,
    report_to="none",
    load_best_model_at_end=True,
    logging_dir="./logs",
)
trainer = FloatLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


print("Train samples:", len(tokenized_datasets["train"]))
print("Validation samples:", len(tokenized_datasets["validation"]))
print("Model device:", next(model.parameters()).device)
trainer.train()


In [ ]:
from sklearn.metrics import f1_score

pred = trainer.predict(tokenized_datasets["validation"])

probs = 1 / (1 + np.exp(-pred.predictions))
labels = pred.label_ids

preds = (probs > 0.25).astype(int)

print("Final micro-F1:", f1_score(labels, preds, average="micro"))
print("Final macro-F1:", f1_score(labels, preds, average="macro"))


In [ ]:
label_id = 0   

print("Emotion:", label_names[label_id])
print(cms[label_id])
